## Stratificed Sampling to Mining Validation - 90% CI and 105 MOE

In [6]:
# -*- coding: utf-8 -*-
"""
Two stratified samples from episodes_enriched_combined.csv:
  A) strata = Primary_Label   -> ...__primary.csv
  B) strata = Driver          -> ...__driver.csv

Outputs per job (saved under OUTPUT_DIR):
  1) allocation_plan_moe10__<job>.csv   : stratum, population(N), allocated(n), realized_moe@p=0.5, weight
  2) stratified_sample_moe10__<job>.csv : sampled rows (ALL columns)
  3) cohort_table__<job>.csv            : compact stratum → population, sample, sample_rate + TOTAL row
"""

import os
import math
import hashlib
import numpy as np
import pandas as pd

# ------------ CONFIG ------------
DATA_PATH  = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V4.2\episodes_enriched_combined.csv"
OUTPUT_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Sample"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONF_LEVEL = 0.90   # 90% CI
MOE_TARGET = 0.10   # ±10 pp
P_WORST    = 0.5
MIN_PER_STRATUM = 0
TOP_UP_STRATA_BY_JOB = {}
BASE_SEED = 20250828

JOBS = [
    (["Primary_Label", "primary_label"], "primary"),
    (["Driver", "driver"],               "driver"),
]
# ---------------------------------


# ---------- stats helpers ----------
def z_for(conf: float) -> float:
    if abs(conf - 0.90) < 1e-12: return 1.645
    if abs(conf - 0.95) < 1e-12: return 1.96
    if abs(conf - 0.99) < 1e-12: return 2.576
    return 1.96

def n_for_moe(N: int, e: float, p: float = 0.5, conf: float = 0.95) -> int:
    if N <= 0: return 0
    z = z_for(conf)
    num = (z**2) * p * (1 - p) * N
    den = (e**2) * (N - 1) + (z**2) * p * (1 - p)
    n = int(math.ceil(num / den))
    return max(0, min(n, N))

def realized_moe_p05(N: int, n: int, conf: float = 0.95) -> float:
    if N <= 1 or n <= 0: return float("inf")
    z = z_for(conf)
    se = math.sqrt(0.25 / n) * math.sqrt((N - n) / (N - 1))
    return z * se


# ---------- misc helpers ----------
def stable_seed(label: str, base_seed: int = BASE_SEED) -> int:
    return (int(hashlib.md5(label.encode("utf-8")).hexdigest()[:8], 16) ^ base_seed) & 0x7FFFFFFF

def choose_first_present(df: pd.DataFrame, candidates) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None

def mk_stratum_label(row, cols):
    parts = []
    for c in cols:
        v = row.get(c)
        s = "" if v is None else str(v).strip()
        if s == "": s = "NULL"
        parts.append(f"{c}={s}")
    return "|".join(parts)


def run_sampler_job(df: pd.DataFrame, strata_cols: list[str], job_key: str):
    # 1) Create stratum id
    df = df.copy()
    df["stratum"] = df.apply(lambda r: mk_stratum_label(r, strata_cols), axis=1)

    # 2) Cohort sizes
    cohort = (
        df.groupby("stratum", dropna=False)
          .size()
          .rename("N")
          .reset_index()
    )

    # 3) Allocations (FPC + CI/MOE target)
    cohort["alloc"] = cohort["N"].apply(lambda N: n_for_moe(int(N), e=MOE_TARGET, p=P_WORST, conf=CONF_LEVEL))

    # 3a) Optional floor
    if MIN_PER_STRATUM and MIN_PER_STRATUM > 0:
        cohort.loc[cohort["N"] > 0, "alloc"] = np.maximum(cohort["alloc"].astype(int), int(MIN_PER_STRATUM))

    # 3b) Optional top-ups
    for s, n_min in (TOP_UP_STRATA_BY_JOB.get(job_key, {}) or {}).items():
        mask = cohort["stratum"] == s
        if mask.any():
            cohort.loc[mask, "alloc"] = np.maximum(cohort.loc[mask, "alloc"].astype(int), int(n_min))

    # 3c) Cap by N
    cohort["alloc"] = cohort[["alloc", "N"]].min(axis=1).astype(int)

    # 4) Realized MOE + weight
    cohort["realized_moe_p05"] = cohort.apply(
        lambda r: realized_moe_p05(int(r["N"]), int(r["alloc"]), conf=CONF_LEVEL), axis=1
    ).round(4)
    cohort["weight"] = np.where(cohort["alloc"] > 0, cohort["N"] / cohort["alloc"], np.nan)

    # 5) Draw reproducible sample
    samples = []
    for _, row in cohort.iterrows():
        s = row["stratum"]
        need = int(row["alloc"])
        if need <= 0:
            continue
        pool = df[df["stratum"] == s]
        if pool.empty:
            continue
        take = min(need, len(pool))
        rs = np.random.RandomState(stable_seed(f"{job_key}|{s}"))
        smp = pool.sample(n=take, replace=False, random_state=rs).copy()
        samples.append(smp)

    sampled_df = pd.concat(samples, ignore_index=True) if samples else df.head(0).copy()

    # 6) Save the detailed allocation plan (PLANNING SHEET)
    plan = (
        cohort.loc[:, ["stratum", "N", "alloc", "realized_moe_p05", "weight"]]
              .rename(columns={"N": "population", "alloc": "allocated"})
              .sort_values(["population", "stratum"], ascending=[False, True], ignore_index=True)
    )

    # 7) Build a compact cohort table (COMPACT REPORT + TOTALS)
    cohort_table = plan.loc[:, ["stratum", "population", "allocated"]].copy()
    cohort_table = cohort_table.rename(columns={"allocated": "sample"})
    # compute sample_rate per stratum (skip TOTAL for now)
    cohort_table["sample_rate"] = cohort_table.apply(
        lambda r: (r["sample"] / r["population"]) if r["population"] else np.nan, axis=1
    ).round(4)

    totals = pd.DataFrame([{
        "stratum": "TOTAL",
        "population": int(cohort_table["population"].sum()),
        "sample":     int(cohort_table["sample"].sum()),
        "sample_rate": round(
            (int(cohort_table["sample"].sum()) / int(cohort_table["population"].sum()))
            if int(cohort_table["population"].sum()) else np.nan, 4
        ),
    }])

    cohort_table = pd.concat([cohort_table.sort_values(["population","stratum"], ascending=[False, True], ignore_index=True),
                              totals],
                             ignore_index=True)

    # 8) Save files
    plan_path         = os.path.join(OUTPUT_DIR, f"allocation_plan_moe10__{job_key}.csv")
    sample_path       = os.path.join(OUTPUT_DIR, f"stratified_sample_moe10__{job_key}.csv")
    cohort_table_path = os.path.join(OUTPUT_DIR, f"cohort_table__{job_key}.csv")

    plan.to_csv(plan_path, index=False, encoding="utf-8")
    sampled_df.to_csv(sample_path, index=False, encoding="utf-8")
    cohort_table.to_csv(cohort_table_path, index=False, encoding="utf-8")

    print(f"[{job_key}] strata: {strata_cols}")
    print(f"[{job_key}] N={int(plan['population'].sum())}, n={int(cohort_table.loc[cohort_table['stratum']!='TOTAL','sample'].sum())}")
    print(f"[{job_key}] saved:\n  {plan_path}\n  {sample_path}\n  {cohort_table_path}\n")


# --------- load & run ---------
df = pd.read_csv(DATA_PATH, low_memory=False).fillna("")

for candidates, job_key in JOBS:
    chosen = next((c for c in candidates if c in df.columns), None)
    if not chosen:
        print(f"[{job_key}] SKIPPED: none of {candidates} present.")
        continue
    run_sampler_job(df, [chosen], job_key)

print("Done.")


[primary] strata: ['Primary_Label']
[primary] N=13088, n=667
[primary] saved:
  C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Sample\allocation_plan_moe10__primary.csv
  C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Sample\stratified_sample_moe10__primary.csv
  C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Sample\cohort_table__primary.csv

[driver] strata: ['Driver']
[driver] N=13088, n=296
[driver] saved:
  C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Sample\allocation_plan_moe10__driver.csv
  C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Sample\stratified_sample_moe10__driver.csv
  C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Sample\cohort_table__driver.csv

Done.


## - Generate the 70/30 sampling from all CCEs

In [12]:
import pandas as pd
import random
from pathlib import Path

# =========================
# CONFIG (V3.1-aligned)
# =========================
INPUT_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0")
INPUT_CSV  = INPUT_DIR / "episodes_enriched_combined.csv"   # list of all CCEs (V3.1)

OUTPUT_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#SAMPLE_SIZE        = 1000    # total size across DEV+TEST - Original Size
SAMPLE_SIZE        = 300    # total size across DEV+TEST
#SEED               = 12345   # fixed seed for reproducibility - Original Seed
SEED               = 1234567   # fixed seed for reproducibility
STRATIFY_BY_REPO   = False   # True = proportional per repo
#DEV_RATIO          = 0.70    # 70% to DEV, 30% to TEST - Original Ratio
DEV_RATIO          = 0.0    # 0% to DEV, 100% to TEST

# =========================
# HELPERS
# =========================
def sample_rows(df_in: pd.DataFrame, n: int, seed: int, stratify: bool) -> pd.DataFrame:
    """Sample n rows from df_in (optionally stratified by repo)."""
    if df_in.empty:
        return df_in
    n = min(n, len(df_in))
    if not stratify or "repo" not in df_in.columns:
        return df_in.sample(n=n, random_state=seed)

    # Proportional by repo
    rng = random.Random(seed)
    parts = []
    total = len(df_in)
    for repo, g in df_in.groupby("repo", sort=False):
        k = max(1, round(n * (len(g) / total)))
        parts.append(g.sample(n=min(k, len(g)), random_state=rng.randint(0, 10**9)))
    out = pd.concat(parts, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=seed)  # trim to exact n
    return out

# =========================
# MAIN
# =========================
def main():
    # 1) Load all CCEs
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    if df.empty:
        raise ValueError("Input CCE list is empty.")
    print(f"[INFO] Loaded CCEs: {len(df):,} rows from {INPUT_CSV}")

    # Preserve original column order to enforce identical schema
    original_cols = df.columns.tolist()

    # 2) Draw overall sample of CCE rows (no column changes)
    sample_all = sample_rows(df, SAMPLE_SIZE, SEED, STRATIFY_BY_REPO)

    # 3) Split into DEV/TEST (70/30) with fixed seed
    sample_all = sample_all.sample(frac=1.0, random_state=SEED).reset_index(drop=True)  # shuffle once
    n_total = len(sample_all)
    n_dev = int(round(n_total * DEV_RATIO))
    n_test = n_total - n_dev

    dev_df  = sample_all.iloc[:n_dev].copy().reindex(columns=original_cols)
    test_df = sample_all.iloc[n_dev:].copy().reindex(columns=original_cols)

    # 4) Save full columns, identical to input (order preserved)
    dev_path  = OUTPUT_DIR / "DEV_Commit_Sample.csv"
    test_path = OUTPUT_DIR / "TEST_Commit_Sample.csv"

    dev_df.to_csv(dev_path, index=False, encoding="utf-8")
    test_df.to_csv(test_path, index=False, encoding="utf-8")

    # Optional sanity checks
    assert dev_df.columns.tolist()  == original_cols, "DEV columns differ from input!"
    assert test_df.columns.tolist() == original_cols, "TEST columns differ from input!"

    print(f"[OK] DEV written : {dev_path}  (rows={len(dev_df):,}, cols={len(original_cols)})")
    print(f"[OK] TEST written: {test_path} (rows={len(test_df):,}, cols={len(original_cols)})")
    print(f"[OK] Total sampled: {n_total:,} (DEV={len(dev_df):,}, TEST={len(test_df):,})")

if __name__ == "__main__":
    main()


[INFO] Loaded CCEs: 12,700 rows from C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0\episodes_enriched_combined.csv
[OK] DEV written : C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\DEV_Commit_Sample.csv  (rows=0, cols=37)
[OK] TEST written: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\TEST_Commit_Sample.csv (rows=300, cols=37)
[OK] Total sampled: 300 (DEV=0, TEST=300)
